# WDL — Module 4 · Session 16 — Catchments and hydrographs
### Reading notebook

| | |
|---|---|
| **Course** | Hydroinformatics in a Changing Climate: Water Data Literacy for All |
| **Department** | Civil Engineering · CHRIST (Deemed to be University) |
| **Instructor** | Dr Arpan Pradhan |
| **Module** | Module 4 — Rivers, Reservoirs and Floods |
| **Session** | Session 16 of 30 |
| **CO** | CO1, CO3 |
| **Topics** | Catchment area; runoff; the shape of a hydrograph; peak and lag |
| **Water hook** | *Where does your rain go?* |

---
> **How to use this notebook.** No coding needed. Read each explanation, then press ▶ on the box below it
> (or click the box and press `Shift + Enter`). Some boxes have sliders, number boxes or dropdowns: change them, then press ▶ again.
> Go to **File → Save a copy in Drive** first, so the notebook is yours.

## Why this session matters
Rain falls on a whole **catchment**: all the land that drains to one point on a river. How big it is, and how much of the rain
runs off, decides how high and how fast the river rises. The graph of river flow over time is a **hydrograph**, the most
important picture in flood work.

In [ ]:
#@title ▶ Step 0 · Get ready (press ▶ once before anything else)
import os, json, warnings
import numpy as np, pandas as pd, requests
import matplotlib.pyplot as plt, matplotlib.dates as mdates
from matplotlib.ticker import MaxNLocator
_orig_bar = plt.Axes.bar
def _bar_int_years(self, x, *a, **k):        # whole-number year ticks on bar charts of years
    out = _orig_bar(self, x, *a, **k)
    try:
        xs = np.asarray(x)
        if xs.dtype.kind in "iu" and xs.min() > 1800: self.xaxis.set_major_locator(MaxNLocator(integer=True))
    except Exception: pass
    return out
plt.Axes.bar = _bar_int_years
from IPython.display import display, Markdown
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.figsize": (11, 4.5), "figure.dpi": 110, "font.size": 11,
                     "axes.titleweight": "bold", "axes.spines.top": False, "axes.spines.right": False})
BLUE, ORANGE, RED, GREY, TEAL, GOLD = "#2C6E9B", "#DD8452", "#C44E52", "#8C8C8C", "#1F7A8C", "#C9962E"
AUTO = os.environ.get("WDL_SELFTEST") == "1"

def ok(msg="Correct!"):  display(Markdown(f"✅ **{msg}**"))
def no(msg="Not quite. Check your working and try again."): display(Markdown(f"❌ {msg}"))
def _num(x):
    try: return float(str(x).replace(",", "").replace("%", "").strip())
    except (TypeError, ValueError): return None
def close(a, b, tol=0.02):
    return a is not None and abs(a - b) <= max(abs(b) * tol, 0.01)
def check_num(ans, exp, tol=0.02, fmt="{:,.1f}", hint=""):
    if AUTO: ans = exp
    v = _num(ans)
    if v is None: no("Type your answer in the box first (numbers only)."); return False
    if close(v, float(exp), tol): ok("Correct: " + fmt.format(exp)); return True
    no("Not quite. " + hint); return False
def check_choice(ans, exp, explain="", hint=""):
    if AUTO: ans = exp
    if ans in ("", "choose"): no("Choose an answer from the dropdown first."); return False
    if ans == exp: ok(("Correct. " + explain).strip()); return True
    no(("Not quite. " + hint).strip()); return False

def make_sample_district():
    """Illustrative, synthetic daily rainfall for a South Indian district, 1981-2024."""
    rng = np.random.default_rng(1981)
    days = pd.date_range("1981-01-01", "2024-12-31", freq="D")
    p_wet = np.array([0.03, 0.03, 0.06, 0.15, 0.30, 0.35, 0.42, 0.42, 0.45, 0.45, 0.28, 0.08])
    mu = np.array([4, 4, 7, 11, 13, 8, 8, 9, 13, 14, 12, 6], float)
    years = np.arange(1981, 2025)
    yf = dict(zip(years, np.exp(rng.normal(0, 0.17, len(years)))))
    m = days.month.values - 1
    t = days.year.values - 1981
    yfac = np.array([yf[y] for y in days.year.values])
    p = np.clip(p_wet[m] * (1 - 0.0025 * t) * yfac ** 0.5, 0, 0.95)
    inten = mu[m] * (1 + 0.005 * t) * yfac ** 0.5
    wet = rng.random(len(days)) < p
    amt = rng.gamma(0.7, inten / 0.7)
    rain = np.round(np.where(wet, amt, 0.0), 1)
    for d, v in [("1991-11-14", 142.0), ("2005-10-24", 186.2), ("2017-08-14", 151.4), ("2022-09-05", 168.9)]:
        rain[days.get_loc(pd.Timestamp(d))] = v
    return pd.Series(rain, index=days, name="rain_mm")

def make_messy_record(sample):
    """Three years of the sample record with typical gauge-record problems added."""
    df = sample["2018":"2020"].rename("rain_mm").reset_index()
    df.columns = ["date", "rain_mm"]
    flag_dates = ["2018-02-03", "2018-05-19", "2018-08-07", "2018-11-25", "2019-01-14",
                  "2019-04-02", "2019-09-18", "2019-12-30", "2020-06-06", "2020-10-11"]
    blank_dates = ["2018-07-21", "2019-03-09", "2020-02-17", "2020-08-28"]
    df.loc[df.date.isin(pd.to_datetime(flag_dates)), "rain_mm"] = -999.0
    df.loc[df.date.isin(pd.to_datetime(blank_dates)), "rain_mm"] = np.nan
    df.loc[df.date == "2018-09-12", "rain_mm"] = 999.9      # typing error
    df.loc[df.date == "2020-04-23", "rain_mm"] = -3.0       # impossible negative
    df = df[~df.date.between("2019-07-01", "2019-07-12")]  # 12 days never recorded
    dup = df[df.date.isin(pd.to_datetime(["2018-06-15", "2019-10-03", "2020-11-20"]))]
    df = pd.concat([df, dup]).sort_values("date", kind="stable").reset_index(drop=True)
    df["date"] = df.date.dt.strftime("%Y-%m-%d")
    return df

SAMPLE_URL = "https://raw.githubusercontent.com/PradhanArpan/waterliteracy/main/data/sample_district_daily_1981_2024.csv"
try:
    sample = pd.read_csv(SAMPLE_URL, parse_dates=["date"], index_col="date")["rain_mm"]
except Exception:
    sample = make_sample_district()   # works offline too: same numbers

def find_place(name):
    """Place name -> (lat, lon, label) using Open-Meteo geocoding. Prefers places in India."""
    parts = [p.strip() for p in str(name).split(",") if p.strip()]
    r = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                     params={"name": parts[0], "count": 10, "language": "en", "format": "json"}, timeout=20)
    res = r.json().get("results", []) or []
    res = [x for x in res if x.get("country_code") == "IN"] or res
    if len(parts) > 1:
        res = [x for x in res if parts[1].lower() in str(x.get("admin1", "")).lower()] or res
    if not res: raise ValueError(f"No place found for '{name}'")
    x = res[0]
    return x["latitude"], x["longitude"], f'{x["name"]}, {x.get("admin1", "")}'.strip(", ")

def fetch_power(lat, lon, start=1981, end=2024):
    """Daily rainfall (mm/day) from NASA POWER (PRECTOTCORR) for one point."""
    r = requests.get("https://power.larc.nasa.gov/api/temporal/daily/point", timeout=90, params={
        "parameters": "PRECTOTCORR", "community": "AG", "latitude": round(lat, 4), "longitude": round(lon, 4),
        "start": f"{start}0101", "end": f"{end}1231", "format": "JSON"})
    r.raise_for_status()
    d = r.json()["properties"]["parameter"]["PRECTOTCORR"]
    s = pd.Series(d, dtype=float)
    s.index = pd.to_datetime(s.index, format="%Y%m%d")
    s[s < -98] = np.nan
    return s.round(1).rename("rain_mm")

def fetch_era5(lat, lon, start="2021-01-01", end="2021-12-31"):
    """Daily rainfall (mm/day) from the ERA5 reanalysis via Open-Meteo."""
    r = requests.get("https://archive-api.open-meteo.com/v1/archive", timeout=60, params={
        "latitude": lat, "longitude": lon, "start_date": start, "end_date": end,
        "daily": "precipitation_sum", "timezone": "Asia/Kolkata"})
    r.raise_for_status()
    d = r.json()["daily"]
    return pd.Series(d["precipitation_sum"], index=pd.to_datetime(d["time"]), dtype=float).rename("rain_mm")

def load_rain(source, place="", lat=0.0, lon=0.0):
    """Returns (daily rainfall Series, label, lat, lon). Falls back to the sample district."""
    if source.startswith("Sample") or AUTO:
        return sample, "Sample district (synthetic)", None, None
    try:
        if not (lat and lon):
            lat, lon, place = find_place(place)
        cache = f"rain_{lat:.3f}_{lon:.3f}.csv"
        if os.path.exists(cache):
            s = pd.read_csv(cache, parse_dates=["date"], index_col="date")["rain_mm"]
        else:
            s = fetch_power(lat, lon)
            s.rename_axis("date").to_csv(cache)
        return s, f"{place} ({lat:.2f}°N, {lon:.2f}°E) · NASA POWER", lat, lon
    except Exception as e:
        display(Markdown(f"⚠️ Could not download data for **{place or 'your place'}** ({type(e).__name__}). "
                         "Using the sample district instead. Check the place name or type latitude and longitude."))
        return sample, "Sample district (synthetic)", None, None

def annual(s):  return s.groupby(s.index.year).sum(min_count=300)
def monsoon(s):
    m = s[s.index.month.isin([6, 7, 8, 9])]
    return m.groupby(m.index.year).sum(min_count=100)
display(Markdown("✅ **Ready.** Tools loaded. Now run **Step 1** to choose your data."))

In [ ]:
#@title ▶ Load the storm-and-river model (press ▶)
PATTERN = np.array([0.10, 0.20, 0.30, 0.20, 0.12, 0.08])     # share of the storm falling in each of 6 hours
def storm_hydrograph(storm_mm=80, area_km2=250, runoff_coeff=0.4, base_flow=5.0, hours=96):
    """Hourly rain (mm) and river flow (m³/s) at the catchment outlet, using a triangular unit hydrograph.
    Illustrative teaching model, not a design tool."""
    t = np.arange(hours)
    rain_h = np.zeros(hours); rain_h[2:8] = storm_mm * PATTERN
    eff = rain_h * runoff_coeff                                  # mm that runs off
    tp = max(2.0, 1.0 + 0.35 * np.sqrt(area_km2))                # hours to peak for 1 mm of runoff
    tb = 2.67 * tp                                               # base time of the unit hydrograph
    qp = 2 * area_km2 * 1000 / (tb * 3600)                       # m³/s per mm of runoff
    uh = np.interp(t, [0, tp, tb], [0, qp, 0], right=0)
    q = np.convolve(eff, uh)[:hours] + base_flow
    return pd.DataFrame({"hour": t, "rain_mm": rain_h, "flow_m3s": q}).set_index("hour")
def summary(h, base_flow=5.0):
    rc = (h.rain_mm * h.index).sum() / h.rain_mm.sum()           # centre of the storm (hour)
    pk = h.flow_m3s.idxmax()
    vol = ((h.flow_m3s - base_flow).clip(lower=0) * 3600).sum() / 1e6
    return {"peak": h.flow_m3s.max(), "peak_hour": pk, "lag": pk - rc, "volume_mcm": vol}
display(Markdown("✅ Model ready."))

---
## Part 1 · A storm becomes a flood wave
Press ▶, then move the sliders. *This is a simple teaching model of a catchment, not a real river.*

In [ ]:
#@title ▶ Storm to hydrograph { run: "auto" }
storm_mm = 80 #@param {type:"slider", min:10, max:250, step:5}
area_km2 = 250 #@param {type:"slider", min:10, max:2000, step:10}
runoff_coeff = 0.4 #@param {type:"slider", min:0.05, max:0.95, step:0.05}
h = storm_hydrograph(storm_mm, area_km2, runoff_coeff, hours=72); s = summary(h)
fig, ax = plt.subplots(figsize=(11, 4.8)); ax2 = ax.twinx()
ax2.bar(h.index, h.rain_mm, color=BLUE, alpha=0.5, width=0.9); ax2.set_ylim(h.rain_mm.max() * 3.5, 0); ax2.set_ylabel("Rain (mm/hour)", color=BLUE)
ax.plot(h.index, h.flow_m3s, color="black", lw=2); ax.set_ylabel("River flow (m³/s)"); ax.set_xlabel("Hours from start")
ax.annotate(f"Peak {s['peak']:,.0f} m³/s", xy=(s["peak_hour"], s["peak"]), xytext=(15, -10), textcoords="offset points")
ax.set_title(f"Lag {s['lag']:.1f} h · runoff volume {s['volume_mcm']:.2f} MCM")
plt.tight_layout(); plt.show()

### Read the hydrograph
- **Rising limb**: flow climbs as runoff arrives. **Peak**: the highest flow. **Falling limb (recession)**: the slow drain-down.
- **Lag time**: hours from the centre of the storm to the peak. Bigger catchments have longer lags.
- **Runoff volume**: the area under the curve above base flow. It equals **runoff coefficient × rain depth × area**.

---
## Part 2 · Check the water balance
Press ▶ to confirm that the volume under the hydrograph matches the rain that ran off.

In [ ]:
#@title ▶ Water balance { run: "auto" }
by_formula = runoff_coeff * storm_mm / 1000 * area_km2 * 1e6 / 1e6
display(Markdown(f"Runoff coefficient × rain × area = {runoff_coeff} × {storm_mm} mm × {area_km2} km² = **{by_formula:.2f} MCM** · "
                 f"area under the hydrograph = **{s['volume_mcm']:.2f} MCM**"))

---
## Part 3 · What changes the flood?

| Change | Peak | Lag |
|---|---|---|
| Bigger storm | Higher | About the same |
| Bigger catchment | Much higher | Longer |
| More paving (higher runoff coefficient) | Higher | About the same in this model; shorter in real cities (drains are fast) |
| Forest and wetlands (lower coefficient) | Lower | Longer |

### ⏭️ Next
Open the **Practice activity** and read a hydrograph yourself.